## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [4]:
sao_URL = "https://www.polyu.edu.hk/sao/"
start_idx, stop_idx = 36450, -900

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    return text

sao_loader = RecursiveUrlLoader(
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"News-and-Events", 
        sao_URL+"About-SAO", 
        sao_URL+"Sitemap", 
        sao_URL+"Search-Result",
        sao_URL+"National-Education",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"Student-Development-Section",
        sao_URL+"student-development-section",
        sao_URL+"Counselling-and-Wellness-Section/PolyU-Asian-Universities-Water-Polo-Invitational-Tournament",
        sao_URL+"Counselling-and-Wellness-Section/Wellness-Centre",
        sao_URL+"Counselling-and-Wellness-Section/Sports-Development",
        sao_URL+"Counselling-and-Wellness-Section/Programmes-and-Activities",
        sao_URL+"Student-Resources-and-Support-Section/Outstanding-Student-Academy",
        sao_URL+"Careers-and-Placement-Section/Gallery-and-Publications",
        sao_URL+"Non-local-Student-Services/Settling-in",
        sao_URL+"Non-local-Student-Services/Event-Highlight",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [5]:
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(doc)

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 157
page_content='           

Quick Access

Start main content

													Home
												

													Careers and Placement
												

													Career Advising and Training Workshops
												

Career Advising and Training Workshops

 Please put at least one valid content allocate to this component.

                                 
                            

Career Advising / Mock Interview Sessions

In this 45-minute one-to-one consultation session, Career Advisors provide you with professional advice on career planning, job-search skills, and interview preparation. As for mock interviews, they may be conducted face to face, online, or on the phone. Submit your CV when registering for the sessions.
Read MoreHide

Register via POSS

                                                    See upcoming training workshops
                                                    

Training Workshops

 Every academic year, CPS, SAO organise

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    chunk.metadata["chunk_id"] = f"{source}_chunk_{i}"

In [7]:
print(chunks[10])

page_content='Quick Access

Start main content

													Home
												

													Counselling and Wellness
												

													Student Counselling
												

													Emergency Support
												

													Non-office hours Emergency Support
												

Non-office hours Emergency Support

Office Hour Emergency Support
Community Helpline
Making Appointment (POSS)
Mental Health Educational Material/Resources

You can seek help from:

PolyU-LINE: Non-office Hour Student Counselling Hotline Service
Tel: 8100-1583

It operates during non-office hours as below:

Days 
Service Hours

 Monday to Friday
6 pm to 9 am

 Saturday, Sunday and public holiday

24 hours round the clock

All phone calls will be answered by the Vital Employee Service Consultancy Christian Family Service Centre.

Hospital's Accident and Emergency Department

The nearest public hospital of our campus is:

Queen Elizabeth Hospital
30 Gascoigne Road, Kowloon, Hong Kong
Tel: 3506-8888

P' meta

### 3. Document Embedding in Chroma

In [8]:
SINGLE = False # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_sao_webpage" if not SINGLE else "vaa_documents"

In [9]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="./chroma_db")
collection = client.get_collection(name=collection_name)

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_18602/3659695745.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


In [10]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 650 chunks into ChromaDB to polyu_sao_webpage


/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_18602/2423177955.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


### 4. Simple Testing

In [11]:
query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: Quick Access

Start main content

													Home
												

													Student Resources and Support
												

													Residential Life
												

													Hall Admission
												

													Admission Policies for Undergraduates
												

Admission Policies for Undergraduates

Admission Policy

General Information on Application

Special Readmission Scheme (SRS)

Readmission Scheme of CURI Residential College (RSCRC)

 
Admission Policy
A. Eligibility
The PolyU Student Halls were established with major funding support from the University Grants Committee (UGC), hence, eligibility to hall residence has to be set with reference to the UGC guidelines. The University has come up with a set of policies to govern admission of students to hall residence. The following groups of students are eligible for hall residence:...
Source: https://www.polyu.edu.hk/sao/Student-Resources-and-Support-Section/Residential-Life/Hall-Admission/Admission-Policies-Und